In [2]:
import sys
import os
import pickle
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict
from rdkit import Chem
import matplotlib.pyplot as plt


# Ensure src is in path to import quartz
sys.path.append(os.path.abspath("src"))

from quartz.data import MoleculeData
from quartz.grids import create_grid, to_dxgrid
from quartz.dataprocessing import (
    calculate_average,
    calculate_boltz_avg,
    calculate_pki_avg,
    pair_overlap,
    grid_abs_stddev,
    grid_stddev,
    grid_variance,
    kmeans_clustering
)
# from quartz.utils import psiwfn_to_fchk

In [ ]:
grids = pickle.load(open("grids.pkl", "rb"))

In [ ]:
# Linux paths
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Major/Ligands/all_ligs").expanduser()
charges_dir = Path("sample_data/RESP_all")
excel_path = Path(
    "~/Documents/Major/Ligands/full_database_ligands.xlsx"
).expanduser()

In [9]:
# Load PDBs from directory and map to Excel Data
ligands_dir = Path("~/Documents/Uni/Major/Ligands/all_ligs").expanduser()
charges_dir = Path("sample_data/FXa_PDB_RESP_charges")
excel_path = Path(
    "~/Documents/Uni/Major/Ligands/full_database_ligands.xlsx"
).expanduser()


In [ ]:
# Read Excel file
print(f"Reading Excel file: {excel_path}")
fx_data = pd.read_excel(excel_path)
# Clean column names by stripping whitespace
fx_data.columns = fx_data.columns.str.strip()

pdb_files = list(ligands_dir.glob("*_out_reduced.pdb"))

mol_data_list = []

print(f"Found {len(pdb_files)} PDB files in {ligands_dir}")

for pdb_file in pdb_files:
    # Extract PDB ID from filename (e.g. 1EZQ_out_reduced.pdb -> 1EZQ)
    pdb_id = pdb_file.name.split("_")[0]
    print(f"Processing PDB ID: {pdb_id}")

    # Path to charge file
    charge_file = charges_dir / f"{pdb_id}_out_reduced.csv"
    # print(f"Looking for charge file: {charge_file}")

    if not charge_file.exists():
        print(f"Charge file not found for {pdb_id}, skipping...")
        continue

    extra = defaultdict(float)

    with open(pdb_file, "r") as f:
        pdb_content = f.readlines()
        for line in pdb_content:
            # print(line)
            if line.startswith("REMARK"):
                # print(line.strip())
                if "minimizedAffinity" in line:
                    extra["minimizedAffinity"] = float(line.split()[-1])
                    # print("Minimized Affinity:", extra['minimizedAffinity'])
                if "CNNscore" in line:
                    extra["CNNscore"] = float(line.split()[-1])
                    # print("CNN Score:", extra['CNNscore'])
                if "CNNaffinity" in line:
                    extra["CNNaffinity"] = float(line.split()[-1])
                    # print("CNN Affinity:", extra['CNNaffinity'])

            if (
                extra["minimizedAffinity"]
                and extra["CNNscore"]
                and extra["CNNaffinity"]
            ):
                break

    # Get pKi and Ki for the PDB ID
    # Assuming column is 'PDB ID' after stripping
    row = fx_data[fx_data["PDB ID"].astype(str).str.strip() == pdb_id]

    if row.empty:
        print(f"No FX Data found for {pdb_id}")
        continue

    # Handle potential duplicates or just take first
    ki = float(row["Ki (nM)"].values[0])
    # print("Ki:", ki)
    if "pKi" in row.columns:
        pki = float(row["pKi"].values[0])
    else:
        pki = -np.log10(ki * 1e-9)  # Convert Ki in nM to pKi
    # print("pKi:", pki)

    # Load Mol from PDB
    mol = Chem.MolFromPDBFile(str(pdb_file), removeHs=False)
    if mol is None:
        print(f"Failed to load PDB: {pdb_file.name}")
        continue


    if not charge_file.exists():
        print(f"Charge file not found for {pdb_id}")
        continue

    charges_df = pd.read_csv(charge_file)

    # Check atom count and charge assignment
    if len(charges_df) != mol.GetNumAtoms():
        print(
            f"Atom count mismatch for {pdb_id}: mol {mol.GetNumAtoms()} vs csv {len(charges_df)}"
        )
        continue

    # Assign RESP charges
    charge_sum = 0.0
    for idx, row_charge in charges_df.iterrows():
        atom = mol.GetAtomWithIdx(idx)
        resp_charge = float(row_charge["RESP"])
        atom.SetProp("RESP", str(resp_charge))
        charge_sum += resp_charge

        # Verify assignment
        if abs(float(atom.GetProp("RESP")) - resp_charge) > 1e-6:
            print(f"Charge assignment failed for {pdb_id} atom {idx}")

    # print(f"Processed {pdb_id}: Total Charge = {charge_sum:.4f}")

    # Create MoleculeData
    md = MoleculeData(
        atomic_numbers=np.array([a.GetAtomicNum() for a in mol.GetAtoms()]),
        coordinates=mol.GetConformer().GetPositions(),
        res={},
        smiles=Chem.MolToSmiles(mol),
        rdmol=mol,
        pdb_code=pdb_id,
        filename=pdb_id,
        ki=ki,
        pki=pki,
        extra=extra,
    )
    mol_data_list.append(md)

print(f"Prepared {len(mol_data_list)} MoleculeData objects")

# Generate grids
print("Generating grids...")
grids = create_grid(mol_data_list, props=["RESP"], size=0.5, threshold=0.0001)

# Save to pkl
output_file = "grids.pkl"
with open(output_file, "wb") as f:
    pickle.dump(grids, f)

# print(f"Saved grids to {output_file}")

# # Save to DX
# print("Saving to DX format...")
# to_dxgrid(grids)
# print("Saved DX files to grids/ directory")

Reading Excel file: C:\Users\friedrich\Documents\Uni\Major\Ligands\full_database_ligands.xlsx
Found 1498 PDB files in C:\Users\friedrich\Documents\Uni\Major\Ligands\all_ligs
Processing PDB ID: 0001
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0001_out_reduced.csv
REMARK minimizedAffinity -9.97466469
REMARK CNNscore 0.965939105
REMARK CNNaffinity 7.95609903
Processing PDB ID: 0002
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0002_out_reduced.csv
REMARK minimizedAffinity -8.88623714
REMARK CNNscore 0.929468334
REMARK CNNaffinity 7.62684917
Processing PDB ID: 0003
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0003_out_reduced.csv
REMARK minimizedAffinity -8.91904163
REMARK CNNscore 0.965632498
REMARK CNNaffinity 7.56074142
Processing PDB ID: 0004
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0004_out_reduced.csv
REMARK minimizedAffinity -8.33714199
REMARK CNNscore 0.88932097
REMARK CNNaffinity 7.4593606
Processing PDB ID: 0005
Looking for 

[12:20:55] Explicit valence for atom # 14 C, 5, is greater than permitted


REMARK minimizedAffinity -8.69044113
REMARK CNNscore 0.746125579
REMARK CNNaffinity 7.55479813
Processing PDB ID: 0340
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0340_out_reduced.csv
REMARK minimizedAffinity -11.2274628
REMARK CNNscore 0.942889988
REMARK CNNaffinity 7.56156826
Processing PDB ID: 0341
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0341_out_reduced.csv
REMARK minimizedAffinity -13.0435543
REMARK CNNscore 0.96935451
REMARK CNNaffinity 7.89111185
Processing PDB ID: 0342
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0342_out_reduced.csv
REMARK minimizedAffinity -13.1815023
REMARK CNNscore 0.974297047
REMARK CNNaffinity 7.81223202
Processing PDB ID: 0343
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0343_out_reduced.csv
REMARK minimizedAffinity -12.6296978
REMARK CNNscore 0.971848249
REMARK CNNaffinity 7.97766399
Processing PDB ID: 0344
Looking for charge file: sample_data\FXa_PDB_RESP_charges\0344_out_reduced.csv
REMARK mini

[12:20:59] Explicit valence for atom # 21 C, 5, is greater than permitted


Processing PDB ID: 4BTU
Looking for charge file: sample_data\FXa_PDB_RESP_charges\4BTU_out_reduced.csv
REMARK minimizedAffinity -10.2295027
REMARK CNNscore 0.893273056
REMARK CNNaffinity 8.04164314
Processing PDB ID: 4Y6D
Looking for charge file: sample_data\FXa_PDB_RESP_charges\4Y6D_out_reduced.csv
REMARK minimizedAffinity -9.31722832
REMARK CNNscore 0.978044391
REMARK CNNaffinity 7.41901684
Processing PDB ID: 4Y71
Looking for charge file: sample_data\FXa_PDB_RESP_charges\4Y71_out_reduced.csv
REMARK minimizedAffinity -9.70598602
REMARK CNNscore 0.89351213
REMARK CNNaffinity 7.32570744
Processing PDB ID: 4Y76
Looking for charge file: sample_data\FXa_PDB_RESP_charges\4Y76_out_reduced.csv
REMARK minimizedAffinity -9.16716099
REMARK CNNscore 0.919566393
REMARK CNNaffinity 7.98365545
Processing PDB ID: 4Y79
Looking for charge file: sample_data\FXa_PDB_RESP_charges\4Y79_out_reduced.csv
REMARK minimizedAffinity -8.48379707
REMARK CNNscore 0.988146484
REMARK CNNaffinity 7.13038635
Processing 

ValueError: cannot reshape array of size 215208 into shape (56,62,60)

In [ ]:
avg = calculate_average(grids, props=["RESP"])
pki_avg = calculate_pki_avg(grids, props=["RESP"])
print(np.min(avg.npGrid["RESP"]), np.min(pki_avg.npGrid["RESP"]))
print(np.max(avg.npGrid["RESP"]), np.max(pki_avg.npGrid["RESP"]))

In [ ]:
boltz_avg_cnnaff = calculate_boltz_avg(
    grids, props=["RESP"], energy_key="CNNaffinity"
)
boltz_avg_cnnscore = calculate_boltz_avg(
    grids, props=["RESP"], energy_key="CNNscore"
)
print(np.min(boltz_avg_cnnscore.npGrid["RESP"]), np.max(boltz_avg_cnnscore.npGrid["RESP"]))

In [ ]:
stddev = grid_stddev(grids, props=["RESP"], means=pki_avg)
abs_stddev = grid_abs_stddev(grids, props=["RESP"], means=pki_avg)
variance = grid_variance(grids, props=["RESP"])
print(np.min(stddev.npGrid["RESP"]), np.max(stddev.npGrid["RESP"]))
print(np.min(abs_stddev.npGrid["RESP"]), np.max(abs_stddev.npGrid["RESP"]))
print(np.min(variance.npGrid["RESP"]), np.max(variance.npGrid["RESP"]))

In [ ]:
sd_grids = np.array([g.npGrid["RESP"].astype(np.float64) for g in grids])
stddev = np.nanstd(sd_grids, axis=0, where=~np.isclose(sd_grids, 0), mean=pki_avg.npGrid["RESP"])

In [ ]:
print(np.min(sd_grids), np.max(sd_grids))
print(np.min(stddev), np.max(stddev))

In [ ]:
to_dxgrid(stddev, filename="resp_stddev.dx")
to_dxgrid(abs_stddev, filename="resp_abs_stddev.dx")
to_dxgrid(variance, filename="resp_variance.dx")

# Boltzmann - CNN Score overlap

In [ ]:
bol_cnnsc_overlap = pair_overlap(boltz_avg_cnnscore, grids)
bol_cnnsc_df = pd.DataFrame(bol_cnnsc_overlap)
bol_cnnsc_df["pKi"] = [grid.mol_data.pki for grid in grids]
bol_cnnsc_df.dropna(inplace=True)
correlation = bol_cnnsc_df["RESP"].corr(bol_cnnsc_df["pKi"])
print("Correlation between Boltzmann CNNscore overlap and pKi:", correlation)
plt.figure()
plt.scatter(bol_cnnsc_df["RESP"], bol_cnnsc_df["pKi"])
plt.xlabel("Overlap")
plt.ylabel("pKi")
plt.title("Boltzmann-CNNscore Overlap vs pKi")
plt.show()

# Weighted pKi overlap

In [ ]:
pki_avg_overlap = pair_overlap(pki_avg, grids)
pki_avg_df = pd.DataFrame(pki_avg_overlap)
pki_avg_df["pKi"] = [grid.mol_data.pki for grid in grids]
pki_avg_df.dropna(inplace=True)
correlation_pki = pki_avg_df["RESP"].corr(pki_avg_df["pKi"])
print("Correlation between pKi average overlap and pKi:", correlation_pki)
plt.figure()
plt.scatter(pki_avg_df["RESP"], pki_avg_df["pKi"])
plt.xlabel("Overlap")
plt.ylabel("pKi")
plt.title("pKi Average Overlap vs pKi")
plt.show()

# Std Dev overlap

In [ ]:
from quartz.dataprocessing import grid_stddev


std_dev = grid_stddev(grids, props=["RESP"], means=pki_avg)
print(np.min(std_dev.npGrid["RESP"]), np.max(std_dev.npGrid["RESP"]))
std_dev_overlap = pair_overlap(std_dev, grids)
std_dev_df = pd.DataFrame(std_dev_overlap)
std_dev_df["pKi"] = [grid.mol_data.pki for grid in grids]
std_dev_df.dropna(inplace=True)
correlation_std = std_dev_df["RESP"].corr(std_dev_df["pKi"])
print(
    "Correlation between Standard Deviation overlap and pKi:", correlation_std
)
plt.figure()
plt.scatter(std_dev_df["RESP"], std_dev_df["pKi"])
plt.xlabel("Overlap")
plt.ylabel("pKi")
plt.title("Standard Deviation vs pKi Overlap")
plt.show()

In [ ]:
to_dxgrid(std_dev, filename="std_dev_RESP")
to_dxgrid(pki_avg, filename="pki_avg_RESP")